# Aerodynamics & Commercial Maneuvers

### 🚧 WIP: Work in progress.

Interactive visualizations / schematics of aerodynamics and commercial maneuvers. 

In [ ]:

# third party packages 
import anywidget
import numpy as np
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

## Helpers

In [ ]:
def calculate_bank_angle(airspeed_knots, turn_radius_feet):
    g = 32.174  # acceleration due to gravity in ft/s²
    airspeed_ft_s = airspeed_knots * 1.68781  # convert knots to ft/s
    bank_angle = np.arctan((airspeed_ft_s ** 2) / (g * turn_radius_feet))
    return np.degrees(bank_angle)  # convert from radians to degrees


def calculate_groundspeed(airspeed_knots, windspeed_knots, relative_wind_deg):
    """Calculate the groundspeed given the airspeed, windspeed, and relative wind direction.
    
    Speeds are converted to ft/s and direction to radians for the calculation, 
    then converted back to knots for the output.

    Args:
        airspeed_knots (float): The airspeed in knots.
        windspeed_knots (float): The windspeed in knots.
        relative_wind_deg (float): The relative wind direction in degrees.

    Returns:
        float: The groundspeed in knots.
    """
    relative_wind_rad = np.radians(relative_wind_deg)
    airspeed_ft_s = airspeed_knots * 1.68781 
    windspeed_ft_s = windspeed_knots * 1.68781
    groundspeed_ft_s = airspeed_ft_s + windspeed_ft_s * np.cos(relative_wind_rad)
    groundspeed_knots = groundspeed_ft_s / 1.68781
    return groundspeed_knots


def calculate_rate_of_turn(airspeed_knots, turn_radius_feet):
    """
    Calculate the rate of turn in degrees per second using airspeed in knots and turn radius in feet.

    Args:
        airspeed_knots (float): The airspeed in knots.
        turn_radius_feet (float): The radius of the turn in feet.

    Returns:
        float: The rate of turn in degrees per second.
    """
    airspeed_ft_s = airspeed_knots * 1.68781  
    rate_of_turn_deg_s = (airspeed_ft_s / turn_radius_feet) * (180 / np.pi)
    return rate_of_turn_deg_s


def calculate_bank_angle_by_rate_of_turn(speed_knots, rate_of_turn_deg_s):
    """
    Calculate the bank angle required for a given airspeed and rate of turn.

    Args:
        speed_knots (float): The airspeed in knots.
        rate_of_turn_deg_s (float): The rate of turn in degrees per second.

    Returns:
        float: The required bank angle in degrees.
    """
    g = 32.174
    speed_ft_s = speed_knots * 1.68781
    bank_angle_rad = np.arctan((rate_of_turn_deg_s * speed_ft_s * np.pi) / (180 * g))
    bank_angle_deg = np.degrees(bank_angle_rad)
    return bank_angle_deg


def calculate_rate_of_turn_by_bank_angle(speed_knots, bank_angle_deg):
    """
    Calculate the rate of turn in degrees per second using speed and bank angle.

    Args:
        speed_knots (float): The speed of the aircraft in knots.
        bank_angle_deg (float): The bank angle in degrees.

    Returns:
        float: The rate of turn in degrees per second.
    """
    g = 32.174
    speed_ft_s = speed_knots * 1.68781
    bank_angle_rad = np.radians(bank_angle_deg)
    rate_of_turn_deg_s = (g * np.tan(bank_angle_rad) / speed_ft_s) * (180 / np.pi)
    return float(rate_of_turn_deg_s)


def calculate_pivotal_altitude(groundspeed_knots, msl_altitude_ft=0.0):
    """
    Calculate the pivotal altitude for eights on pylons, adjusted for specified MSL altitude.

    Args:
        groundspeed_knots (float): The groundspeed of the aircraft in knots.
        msl_altitude_ft (float): The Mean Sea Level (MSL) altitude in feet (default: 0.0).

    Returns:
        float: The adjusted pivotal altitude in feet.
    """
    pivotal_altitude_ft = (groundspeed_knots ** 2) / 11.3 + msl_altitude_ft
    return pivotal_altitude_ft

### Bank Angle

The bank angle $ \theta $ can be calculated using the formula:

$ \theta = \tan^{-1} \left( \frac{v^2}{g \cdot r} \right) $

where:
- $\theta$: Bank angle (in degrees) 
- $v$: Airspeed (in feet per second) $ v = \text{airspeed\_knots} \times 1.68781 $
- $g$: Acceleration due to gravity ($ 32.174 \, \text{ft/s}^2 $)
- $r$: Turn radius (in feet)

In [ ]:
# Bank Angle Calculator with Plotly Gauge

# Create gauge
bank_gauge = go.FigureWidget(
    go.Indicator(
        mode="gauge+number",
        value=calculate_bank_angle(90, 1250),
        number={'suffix': '°', 'font': {'size': 40}},
        gauge={
            'axis': {'range': [0, 60], 'tickwidth': 1, 'tickcolor': "#333"},
            'bar': {'color': "#1f77b4"},
            'bgcolor': "white",
            'borderwidth': 2,
            'bordercolor': "#ccc",
            'steps': [
                {'range': [0, 20], 'color': '#e8f4f8'},
                {'range': [20, 30], 'color': '#b8d4e3'},
                {'range': [30, 45], 'color': '#ffeaa7'},
                {'range': [45, 60], 'color': '#fab1a0'}
            ],
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': 45
            }
        },
        title={'text': "Bank Angle", 'font': {'size': 20}}
    )
)
bank_gauge.update_layout(
    template='plotly_white',
    height=350,
    margin=dict(l=20, r=20, t=80, b=20)
)

# Create sliders
airspeed_slider_bank = widgets.FloatSlider(
    value=90, min=50, max=150, step=1,
    description='Airspeed (kts):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
turn_radius_slider_bank = widgets.FloatSlider(
    value=1250, min=100, max=5280, step=10,
    description='Turn Radius (ft):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Update callback
def update_bank_gauge(change):
    bank_angle = calculate_bank_angle(
        airspeed_slider_bank.value,
        turn_radius_slider_bank.value
    )
    with bank_gauge.batch_update():
        bank_gauge.data[0].value = bank_angle

airspeed_slider_bank.observe(update_bank_gauge, names='value')
turn_radius_slider_bank.observe(update_bank_gauge, names='value')

# Display sliders and figure separately (VS Code workaround)
display(widgets.VBox([
    airspeed_slider_bank, 
    turn_radius_slider_bank
], layout=widgets.Layout(margin='0 0 20px 0')))
display(bank_gauge)

&nbsp;

### Rate of Turn

The rate of turn (ROT) in degrees per second is calculated as:

$
\text{ROT (deg/s)} = \frac{\text{Speed (ft/s)}}{\text{Turn Radius (ft)}} \times \frac{180}{\pi}
$

Where:
- $Speed$: is the airspeed (ft/s).
- $Turn Radius$: is the radius of the turn (ft).
- $\frac{180}{\pi}$: converts the angular velocity from radians per second to degrees per second.

In [ ]:
# Rate of Turn Calculator with Plotly Gauge

def create_rot_gauge(rate_of_turn):
    """Create a gauge figure showing rate of turn."""
    fig = go.FigureWidget(
        go.Indicator(
            mode="gauge+number",
            value=rate_of_turn,
            number={'suffix': '°/s', 'font': {'size': 40}},
            gauge={
                'axis': {'range': [0, 10], 'tickwidth': 1, 'tickcolor': "#333"},
                'bar': {'color': "#2ecc71"},
                'bgcolor': "white",
                'borderwidth': 2,
                'bordercolor': "#ccc",
                'steps': [
                    {'range': [0, 3], 'color': '#e8f8f0'},
                    {'range': [3, 6], 'color': '#b8e8d0'},
                    {'range': [6, 10], 'color': '#ffeaa7'}
                ],
                'threshold': {
                    'line': {'color': "orange", 'width': 4},
                    'thickness': 0.75,
                    'value': 3  # Standard rate turn marker
                }
            },
            title={'text': "Rate of Turn", 'font': {'size': 20}}
        )
    )
    fig.update_layout(
        template='plotly_white',
        height=350,
        margin=dict(l=20, r=20, t=80, b=20),
        annotations=[dict(
            x=0.5, y=-0.1,
            text="Standard rate turn = 3°/s",
            showarrow=False,
            font=dict(size=12, color="#666")
        )]
    )
    return fig

# Create initial gauge
initial_rot = calculate_rate_of_turn(90, 1250)
rot_gauge = create_rot_gauge(initial_rot)

# Create sliders
airspeed_slider_rot = widgets.FloatSlider(
    value=90, min=50, max=150, step=1,
    description='Airspeed (kts):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
turn_radius_slider_rot = widgets.FloatSlider(
    value=1250, min=100, max=5280, step=10,
    description='Turn Radius (ft):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

def update_rot_gauge(change):
    rate_of_turn = calculate_rate_of_turn(
        airspeed_slider_rot.value,
        turn_radius_slider_rot.value
    )
    with rot_gauge.batch_update():
        rot_gauge.data[0].value = rate_of_turn

airspeed_slider_rot.observe(update_rot_gauge, names='value')
turn_radius_slider_rot.observe(update_rot_gauge, names='value')

# Display sliders and figure separately (VS Code workaround)
display(widgets.VBox([
    airspeed_slider_rot,
    turn_radius_slider_rot
], layout=widgets.Layout(margin='0 0 20px 0')))
display(rot_gauge)

&nbsp;

### Groundspeed

In [ ]:
# Groundspeed Visualization with Plotly

relative_wind_deg = np.linspace(0, 180, 100)

def compute_groundspeeds(airspeed, windspeed):
    return [calculate_groundspeed(airspeed, windspeed, angle) for angle in relative_wind_deg]

# Create initial figure
initial_gs = compute_groundspeeds(90, 10)

gs_fig = go.FigureWidget(
    data=[
        go.Scatter(
            x=relative_wind_deg,
            y=initial_gs,
            mode='lines',
            name='Groundspeed',
            line=dict(color='#3498db', width=3),
            hovertemplate='Wind Angle: %{x:.0f}°<br>Groundspeed: %{y:.1f} kts<extra></extra>'
        )
    ],
    layout=go.Layout(
        title='Groundspeed vs Relative Wind Direction',
        xaxis=dict(title='Relative Wind Direction (°)', range=[0, 180]),
        yaxis=dict(title='Groundspeed (knots)'),
        template='plotly_white',
        height=400,
        margin=dict(l=60, r=20, t=60, b=60),
        hovermode='x unified'
    )
)

# Create sliders
airspeed_slider_gs = widgets.FloatSlider(
    value=90, min=50, max=300, step=1,
    description='Airspeed (kts):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
windspeed_slider_gs = widgets.FloatSlider(
    value=10, min=0, max=100, step=1,
    description='Windspeed (kts):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

def update_gs_plot(change):
    new_gs = compute_groundspeeds(airspeed_slider_gs.value, windspeed_slider_gs.value)
    with gs_fig.batch_update():
        gs_fig.data[0].y = new_gs

airspeed_slider_gs.observe(update_gs_plot, names='value')
windspeed_slider_gs.observe(update_gs_plot, names='value')

# Display sliders and figure separately (VS Code workaround)
display(widgets.VBox([
    airspeed_slider_gs,
    windspeed_slider_gs
], layout=widgets.Layout(margin='0 0 20px 0')))
display(gs_fig)

## Chandelles


Phase 1 (0-90°): Establish 30° bank, smooth pitch up to max.  
Phase 2 (90-180°): Maintain constant pitch, smooth roll-out to level.

In [ ]:
def generate_bank_angle(turn_deg):
    """Smooth transition to 30°, hold until 90°, then roll out."""
    bank_angle = np.zeros_like(turn_deg, dtype=float)
    for i, turn in enumerate(turn_deg):
        if turn <= 15:
            # Quick but smooth entry to bank
            bank_angle[i] = 30 * np.sin(np.radians((turn/15) * 90))
        elif turn <= 90:
            # Constant bank phase
            bank_angle[i] = 30
        else:
            # Roll out phase: Linear roll-out is standard for Chandelles
            bank_angle[i] = 30 - (30 / 90) * (turn - 90)
    return bank_angle

def generate_pitch_angle(turn_deg):
    """Smoothly increase pitch to 15° at the 90° mark, then hold constant."""
    pitch_angle = np.zeros_like(turn_deg, dtype=float)
    for i, turn in enumerate(turn_deg):
        if turn <= 90:
            # Sinusoidal pitch increase to avoid 'linear' feel
            pitch_angle[i] = 15 * np.sin(np.radians((turn/90) * 90))
        else:
            # Constant pitch phase
            pitch_angle[i] = 15
    return pitch_angle

def generate_airspeed_profile(turn_deg, airspeed_start=90, vs1=50):
    """
    Non-linear airspeed decay. The 'drag hole' is most significant 
    in the final 45° as airspeed approaches stall.
    """
    airspeed = np.zeros_like(turn_deg, dtype=float)
    airspeed_end = vs1 + 5 # Target minimum speed at the end
    
    # Using a power-law to simulate induced drag peaking at the end
    for i, turn in enumerate(turn_deg):
        progress = turn / 180
        # Exponent > 1 causes the speed to 'fall off' faster at the end
        airspeed[i] = airspeed_start - (airspeed_start - airspeed_end) * (progress ** 1.4)
    return airspeed

# --- Execution ---
turn_deg = np.linspace(0, 180, 181)
bank_angle = generate_bank_angle(turn_deg)
pitch_angle = generate_pitch_angle(turn_deg)
# Using our physics-based decay
airspeed = generate_airspeed_profile(turn_deg, airspeed_start=90, vs1=50)

# Calculate Load Factor (G)
# Load factor in a Chandelle is n = 1/cos(bank) + a bit of pitch pull
load_factor = (1 / np.cos(np.radians(bank_angle))) + (0.15 * np.sin(np.radians(turn_deg/2)))

# --- Visualization ---
chandelle_fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=('Bank Angle', 'Pitch Angle', 'Airspeed', 'Load Factor'),
    horizontal_spacing=0.06
)

# Add traces (consistent with your previous style)
chandelle_fig.add_trace(go.Scatter(x=turn_deg, y=bank_angle, name='Bank', line=dict(color='#3498db')), row=1, col=1)
chandelle_fig.add_trace(go.Scatter(x=turn_deg, y=pitch_angle, name='Pitch', line=dict(color='#e67e22')), row=1, col=2)
chandelle_fig.add_trace(go.Scatter(x=turn_deg, y=airspeed, name='Airspeed', line=dict(color='#2ecc71')), row=1, col=3)
chandelle_fig.add_trace(go.Scatter(x=turn_deg, y=load_factor, name='G-Load', line=dict(color='#e74c3c')), row=1, col=4)

# Reference at 90°
for col in [1, 2, 3, 4]:
    chandelle_fig.add_vline(x=90, line_dash="dash", line_color="gray", row=1, col=col)

chandelle_fig.update_layout(title_text="Physics-Refined Chandelle Profile", template='plotly_white', height=400)
chandelle_fig.show()

In [ ]:
# %% [markdown]
# ## Chandelle: Rate of Turn and Turn Radius
#
# This block calculates the geometry of the turn. Note that as airspeed
# drops, the aircraft's turn performance changes dynamically.

# %%
# Calculate rate of turn: (1091 * tan(bank)) / V
rate_of_turn = [
    calculate_rate_of_turn_by_bank_angle(as_val, ba_val) if as_val > 0 else 0
    for as_val, ba_val in zip(airspeed, bank_angle)
]

# Calculate turn radius: V^2 / (11.26 * tan(bank)) 
# Or using your method: (V_fps) / (ROT_rad_s)
turn_radius = [
    (as_val * 1.68781) / ((rot * np.pi) / 180) if rot > 0.1 else np.nan
    for as_val, rot in zip(airspeed, rate_of_turn)
]

# Create subplots
rot_radius_fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Rate of Turn', 'Turn Radius (Dynamic Spiral)'),
    horizontal_spacing=0.12
)

rot_radius_fig.add_trace(
    go.Scatter(
        x=turn_deg, y=rate_of_turn,
        mode='lines', name='Rate of Turn',
        line=dict(color='#9b59b6', width=2.5),
        hovertemplate='Turn: %{x:.0f}°<br>ROT: %{y:.2f}°/s<extra></extra>'
    ),
    row=1, col=1
)

# We show the full range but the radius will naturally clip as it goes to infinity
rot_radius_fig.add_trace(
    go.Scatter(
        x=turn_deg, y=turn_radius,
        mode='lines', name='Turn Radius',
        line=dict(color='#e67e22', width=2.5),
        hovertemplate='Turn: %{x:.0f}°<br>Radius: %{y:.0f} ft<extra></extra>'
    ),
    row=1, col=2
)

# Add 90-degree marker
for col in [1, 2]:
    rot_radius_fig.add_vline(x=90, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=col)

rot_radius_fig.update_xaxes(title_text="Turn Degrees", row=1, col=1)
rot_radius_fig.update_xaxes(title_text="Turn Degrees", range=[0, 180], row=1, col=2)
rot_radius_fig.update_yaxes(title_text="Degrees / Second", row=1, col=1)
rot_radius_fig.update_yaxes(title_text="Feet", range=[0, 2500], row=1, col=2) # Clamped for scannability

rot_radius_fig.update_layout(
    template='plotly_white',
    height=450,
    width=900,
    title_text="Chandelle Geometry: Rate vs. Radius",
    hovermode='x unified'
)

rot_radius_fig.show()

In [ ]:
# %% [markdown]
# ## Chandelle: Ground Track (X-Y Coordinates)
#
# This demonstrates the "tightening spiral" effect. Because airspeed decays 
# significantly, the second 90 degrees of turn covers much less ground 
# than the first 90 degrees.

# %%
def generate_ground_track(turn_deg, airspeed_knots):
    """
    Calculates X and Y coordinates in feet.
    Assumes no wind for a pure aerodynamic profile.
    """
    # Convert knots to feet per second
    v_fps = airspeed_knots * 1.68781
    
    # Initialize coordinates
    x = np.zeros_like(turn_deg)
    y = np.zeros_like(turn_deg)
    
    # Time per degree (dt) is 1 / Rate of Turn
    # Using a simple cumulative sum of vectors
    # Heading starts at 0 (North) and goes to 180 (South)
    for i in range(1, len(turn_deg)):
        # Calculate heading in radians
        heading_rad = np.radians(turn_deg[i])
        
        # Determine the 'step size' based on average velocity between points
        # Assuming we move 1 degree of turn, the distance is roughly turn_radius * d_theta
        # But we'll use a velocity-time step for accuracy
        rot = calculate_rate_of_turn_by_bank_angle(airspeed_knots[i], bank_angle[i])
        
        if rot > 0:
            dt = 1.0 / rot # seconds to turn 1 degree
            distance = v_fps[i] * dt
            
            # Update position (standard 2D rotation)
            x[i] = x[i-1] + distance * np.sin(heading_rad)
            y[i] = y[i-1] + distance * np.cos(heading_rad)
        else:
            # If wings are level, just move straight
            x[i] = x[i-1]
            y[i] = y[i-1] + v_fps[i] * 0.1 # nominal step

    return x, y

# --- Execute Ground Track ---
x_coords, y_coords = generate_ground_track(turn_deg, airspeed)

# --- Visualization ---
track_fig = go.Figure()

# Plot the path
track_fig.add_trace(go.Scatter(
    x=x_coords, y=y_coords,
    mode='lines+markers',
    name='Chandelle Track',
    line=dict(color='#2c3e50', width=4),
    marker=dict(
        size=4,
        color=turn_deg, # Color by degrees to show progress
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Turn Deg")
    ),
    hovertemplate='X: %{x:.0f} ft<br>Y: %{y:.0f} ft<br>Turn: %{marker.color}°'
))

# Annotate key points
points = {0: 'Entry', 90: 'Max Pitch', 180: 'Exit'}
for deg, label in points.items():
    idx = int(deg)
    track_fig.add_annotation(
        x=x_coords[idx], y=y_coords[idx],
        text=label, showarrow=True, arrowhead=2,
        ax=40 if x_coords[idx] > 0 else -40, ay=-40
    )

track_fig.update_layout(
    title="Chandelle Ground Track (Zero Wind)",
    xaxis_title="East / West Distance (feet)",
    yaxis_title="North / South Distance (feet)",
    template='plotly_white',
    height=600,
    width=600,
    yaxis=dict(scaleanchor="x", scaleratio=1), # Keep aspect ratio 1:1
    showlegend=False
)

track_fig.show()


## Lazy Eights


The Lazy Eight is a continuous series of climbing and descending turns in the shape of a figure 8. The maneuver consists of two symmetrical halves (0-180° and 180-360°).

**Mechanics:**
- **0-45°**: Increase pitch to max ~15° at 45°, increase bank from 0° to ~15°
- **45-90°**: Decrease pitch to 0° at 90°, increase bank from 15° to 30°
- **90-135°**: Decrease pitch to -15° at 135°, decrease bank from 30° to 15°
- **135-180°**: Increase pitch to 0° at 180°, decrease bank from 15° to 0°

**Constraints:** Same altitude and airspeed at entry (0°) and exit (180°).


In [ ]:

# %%
# Lazy Eight Profile Functions

def calculate_rate_of_turn_by_bank_angle(airspeed, bank_angle):
    if airspeed < 1: return 0
    # standard aviation formula: rate = (1091 * tan(bank)) / airspeed
    return (1091 * np.tan(np.radians(bank_angle))) / airspeed

def generate_lazy_eight_bank_angle(turn_deg):
    """Sinusoidal bank profile: Smoothly peaks at 30° at the 90° point."""
    return 30 * np.sin(np.radians(turn_deg))

def generate_lazy_eight_pitch_angle(turn_deg):
    """Sinusoidal pitch profile: Max pitch at 45°, through horizon at 90°."""
    return 15 * np.sin(np.radians(turn_deg * 2))

def generate_lazy_eight_airspeed(turn_deg, entry_airspeed=90, vs1=48):
    """
    Simulates non-linear airspeed decay. 
    Uses a power-law to model exponential induced drag at high AOA.
    """
    airspeed = np.zeros_like(turn_deg, dtype=float)
    min_speed = vs1 + 7  # 5-10 knots above stall
    max_speed = entry_airspeed + 5 

    for i, turn in enumerate(turn_deg):
        if turn <= 90:
            # Power law (1.6) models the rapid bleed-off near the 90° point
            progress = turn / 90
            airspeed[i] = entry_airspeed - (entry_airspeed - min_speed) * (progress ** 1.6)
        elif turn <= 135:
            # Gravity assist acceleration as the nose falls through the horizon
            progress = (turn - 90) / 45
            airspeed[i] = min_speed + (max_speed - min_speed) * (progress ** 0.8)
        else:
            # Returning to entry speed and altitude at 180°
            progress = (turn - 135) / 45
            airspeed[i] = max_speed - (max_speed - entry_airspeed) * progress
    return airspeed

def generate_lazy_eight_load_factor(turn_deg, bank_angles, pitch_angles):
    """
    Models the asymmetrical G-loading.
    The pull-out (135-180) is 'heavier' than the initial pull-up (0-45).
    """
    load_factor = np.zeros_like(turn_deg, dtype=float)
    for i, turn in enumerate(turn_deg):
        # Static G from bank
        n_bank = 1 / np.cos(np.radians(bank_angles[i]))
        
        # Dynamic G from pitch changes
        if turn <= 45:
            # Initial pull-up: light and smooth
            pitch_g = 0.2 * np.sin(np.radians(turn * 2))
        elif turn <= 135:
            # The 'Push-over' and 'Drop': G drops below 1.0 at the 90 crest
            # Using a deeper sine dip to model the weightlessness
            pitch_g = 0.25 * np.sin(np.radians(turn * 2)) - 0.15 * np.exp(-((turn-90)**2)/400)
        else:
            # The Pull-out (135-180): High airspeed + high rate of pitch change
            # This is the 'heavy' part of the maneuver.
            progress = (turn - 135) / 45
            pitch_g = 0.45 * np.cos(np.radians((turn - 180) * 2))
            
        load_factor[i] = n_bank + pitch_g
    return load_factor

def generate_lazy_eight_vsi(turn_deg, airspeed_knots, pitch_angles):
    """
    Calculates a realistic VSI by estimating Flight Path Angle (gamma).
    Ensures VSI is 0 at entry (0°), crest (90°), and exit (180°).
    """
    v_fps = airspeed_knots * 1.68781
    vsi_fpm = np.zeros_like(turn_deg, dtype=float)
    
    for i, turn in enumerate(turn_deg):
        # We model Flight Path Angle as a fraction of Pitch Angle 
        # to account for the high AOA/mushing at low speeds.
        # At the 90-degree point, pitch is 0, so VSI will naturally be 0.
        
        if turn <= 90:
            # During the climb, the flight path is roughly 40-50% of pitch
            gamma = pitch_angles[i] * 0.40 
        else:
            # During descent, the flight path is closer to pitch as speed builds
            gamma = pitch_angles[i] * 0.52
            
        vsi_fpm[i] = v_fps[i] * np.sin(np.radians(gamma)) * 60
        
    return vsi_fpm
# %%
# Generate Lazy Eight data
turn_deg_le = np.linspace(0, 180, 181)
bank_angle_le = generate_lazy_eight_bank_angle(turn_deg_le)
pitch_angle_le = generate_lazy_eight_pitch_angle(turn_deg_le)
airspeed_le = generate_lazy_eight_airspeed(turn_deg_le, entry_airspeed=90, vs1=50)


# Create subplots
lazy_eight_fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Bank Angle', 'Pitch Angle', 'Airspeed'),
    horizontal_spacing=0.08
)

lazy_eight_fig.add_trace(
    go.Scatter(
        x=turn_deg_le, y=bank_angle_le,
        mode='lines', name='Bank Angle',
        line=dict(color='#3498db', width=2),
        hovertemplate='Turn: %{x:.0f}°<br>Bank: %{y:.1f}°<extra></extra>'
    ),
    row=1, col=1
)

lazy_eight_fig.add_trace(
    go.Scatter(
        x=turn_deg_le, y=pitch_angle_le,
        mode='lines', name='Pitch Angle',
        line=dict(color='#e67e22', width=2),
        hovertemplate='Turn: %{x:.0f}°<br>Pitch: %{y:.1f}°<extra></extra>'
    ),
    row=1, col=2
)

lazy_eight_fig.add_trace(
    go.Scatter(
        x=turn_deg_le, y=airspeed_le,
        mode='lines', name='Airspeed',
        line=dict(color='#2ecc71', width=2),
        hovertemplate='Turn: %{x:.0f}°<br>Airspeed: %{y:.1f} kts<extra></extra>'
    ),
    row=1, col=3
)

# Add vertical reference lines at key points
key_points = [45, 90, 135]
for col in [1, 2, 3]:
    for point in key_points:
        lazy_eight_fig.add_vline(
            x=point, line_dash="dash", line_color="gray", opacity=0.5,
            row=1, col=col
        )

# Add horizontal reference line at pitch = 0
lazy_eight_fig.add_hline(
    y=0, line_dash="dot", line_color="red", opacity=0.3,
    row=1, col=2
)

lazy_eight_fig.update_xaxes(title_text="Turn Degrees", row=1, col=1)
lazy_eight_fig.update_xaxes(title_text="Turn Degrees", row=1, col=2)
lazy_eight_fig.update_xaxes(title_text="Turn Degrees", row=1, col=3)
lazy_eight_fig.update_yaxes(title_text="Bank Angle (°)", row=1, col=1)
lazy_eight_fig.update_yaxes(title_text="Pitch Angle (°)", row=1, col=2)
lazy_eight_fig.update_yaxes(title_text="Airspeed (knots)", row=1, col=3)

lazy_eight_fig.update_layout(
    template='plotly_white',
    height=400,
    width=1000,
    title_text="Lazy Eight Maneuver Profile (First Half: 0-180°)",
    showlegend=False,
    hovermode='x unified'
)

lazy_eight_fig.show()


In [ ]:

# ## Lazy Eight: Rate of Turn, Load Factor, and Vertical Speed
# 
# This block integrates the horizontal performance (ROT), the structural 
# loading (G), and the vertical performance (VSI) into a single dashboard.

# %%
# Calculate rate of turn throughout the maneuver
rate_of_turn_le = [
    calculate_rate_of_turn_by_bank_angle(as_val, ba_val) if ba_val > 0 else 0
    for as_val, ba_val in zip(airspeed_le, bank_angle_le)
]

# Calculate load factor with pitch dynamics
load_factor_le = generate_lazy_eight_load_factor(turn_deg_le, bank_angle_le, pitch_angle_le)

# Calculate vertical speed (Feet Per Minute)
vsi_le = generate_lazy_eight_vsi(turn_deg_le, airspeed_le, pitch_angle_le)

# Create subplots (1 row, 3 columns)
rot_load_fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Rate of Turn', 'Load Factor', 'Vertical Speed (VSI)'),
    horizontal_spacing=0.1
)

# 1. Rate of Turn Trace
rot_load_fig.add_trace(
    go.Scatter(
        x=turn_deg_le, y=rate_of_turn_le,
        mode='lines', name='Rate of Turn',
        line=dict(color='#9b59b6', width=2),
        hovertemplate='Turn: %{x:.0f}°<br>ROT: %{y:.2f}°/s<extra></extra>'
    ),
    row=1, col=1
)

# 2. Load Factor Trace
rot_load_fig.add_trace(
    go.Scatter(
        x=turn_deg_le, y=load_factor_le,
        mode='lines', name='Load Factor',
        line=dict(color='#e74c3c', width=2),
        hovertemplate='Turn: %{x:.0f}°<br>G-Load: %{y:.2f}G<extra></extra>'
    ),
    row=1, col=2
)

# 3. Vertical Speed Trace
rot_load_fig.add_trace(
    go.Scatter(
        x=turn_deg_le, y=vsi_le,
        mode='lines', name='VSI',
        line=dict(color='#2c3e50', width=2),
        fill='tozeroy',  # Visualizes climb vs descent relative to zero
        hovertemplate='Turn: %{x:.0f}°<br>VSI: %{y:.0f} fpm<extra></extra>'
    ),
    row=1, col=3
)

# Add reference lines and annotations
key_points = [45, 90, 135]
for col in [1, 2, 3]:
    for point in key_points:
        rot_load_fig.add_vline(x=point, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=col)

# Add horizontal reference at 1G for Load Factor
rot_load_fig.add_hline(
    y=1.0, line_dash="dot", line_color="green", opacity=0.5,
    annotation_text="1G", annotation_position="right",
    row=1, col=2
)

# Add horizontal reference at 0 FPM for VSI
rot_load_fig.add_hline(
    y=0, line_dash="dot", line_color="black", opacity=0.5,
    row=1, col=3
)

# Add shaded region for typical pull zone (G-Load)
rot_load_fig.add_hrect(
    y0=1.1, y1=1.4, 
    fillcolor="yellow", opacity=0.1,
    annotation_text="Pull-out Zone", annotation_position="top left",
    row=1, col=2
)

# Axis Updates
rot_load_fig.update_xaxes(title_text="Turn Degrees", row=1, col=1)
rot_load_fig.update_xaxes(title_text="Turn Degrees", row=1, col=2)
rot_load_fig.update_xaxes(title_text="Turn Degrees", row=1, col=3)

rot_load_fig.update_yaxes(title_text="°/sec", row=1, col=1)
rot_load_fig.update_yaxes(title_text="G", range=[0.5, 1.8], row=1, col=2)
rot_load_fig.update_yaxes(title_text="Feet/Min", row=1, col=3)

rot_load_fig.update_layout(
    template='plotly_white',
    height=450,
    width=1100,
    title_text="Lazy Eight Performance Dashboard",
    showlegend=False,
    hovermode='x unified'
)

rot_load_fig.show()

#### Lazy Eight: Ground Track (X-Y Coordinates)  
This demonstrates the symmetry (or lack thereof) in the ground track. The loop appears 'compressed' at the 90° point due to low airspeed and 'elongated' at the 180° point due to high airspeed.

In [ ]:

# %%
def generate_lazy_eight_track(turn_deg, airspeed_knots, bank_angles):
    """
    Calculates X and Y coordinates for a 180-degree half of a Lazy Eight.
    """
    v_fps = airspeed_knots * 1.68781
    x = np.zeros_like(turn_deg)
    y = np.zeros_like(turn_deg)
    
    # We assume we start at (0,0) flying along the Y-axis (Heading 000)
    # The turn_deg represents our cumulative change in heading.
    for i in range(1, len(turn_deg)):
        heading_rad = np.radians(turn_deg[i])
        
        # Calculate Rate of Turn to find time spent at this degree
        rot = calculate_rate_of_turn_by_bank_angle(airspeed_knots[i], bank_angles[i])
        
        if rot > 0.1:
            dt = 1.0 / rot 
            distance = v_fps[i] * dt
            
            # Update position
            # Sin/Cos are swapped here to show a turn starting from a North heading
            x[i] = x[i-1] + distance * np.sin(heading_rad)
            y[i] = y[i-1] + distance * np.cos(heading_rad)
        else:
            # Handling level flight / transition points
            x[i] = x[i-1]
            y[i] = y[i-1] + v_fps[i] * 0.1

    return x, y

# --- Execute Ground Track ---
# Using the previously defined airspeed_le and bank_angle_le from the Lazy Eight block
x_le, y_le = generate_lazy_eight_track(turn_deg_le, airspeed_le, bank_angle_le)

# --- Visualization ---
lazy_track_fig = go.Figure()

lazy_track_fig.add_trace(go.Scatter(
    x=x_le, y=y_le,
    mode='lines',
    name='Lazy Eight Path',
    line=dict(color='#16a085', width=4),
    hovertemplate='X: %{x:.0f} ft<br>Y: %{y:.0f} ft'
))

# Highlight the 90-degree point (Top of the Eight)
lazy_track_fig.add_trace(go.Scatter(
    x=[x_le[90]], y=[y_le[90]],
    mode='markers+text',
    name='90° Point',
    text=['90° (Min Speed)'],
    textposition="top center",
    marker=dict(color='red', size=10)
))

lazy_track_fig.update_layout(
    title="Lazy Eight Ground Track (First 180° Turn)",
    xaxis_title="East / West (feet)",
    yaxis_title="North / South (feet)",
    template='plotly_white',
    height=600,
    width=600,
    yaxis=dict(scaleanchor="x", scaleratio=1),
    showlegend=False
)

lazy_track_fig.show()

## Steep Spiral

This is a ground reference maneuver establishing a descending constant airspeed constant radius turn. So we'll use groundspeed to calculate the bank angle as a function of the airspeed, relative wind and desired radius of the turn.

In [ ]:
# Steep Spiral Visualization with Plotly

relative_wind_deg_ss = np.linspace(0, 180, 100)

def compute_steep_spiral_data(airspeed, windspeed, turn_radius):
    groundspeeds = [calculate_groundspeed(airspeed, windspeed, angle) for angle in relative_wind_deg_ss]
    bank_angles = [calculate_bank_angle(gs, turn_radius) for gs in groundspeeds]
    return groundspeeds, bank_angles

# Create initial figure
initial_gs_ss, initial_ba_ss = compute_steep_spiral_data(90, 10, 1250)
max_diff = max(initial_ba_ss) - min(initial_ba_ss)

ss_fig = go.FigureWidget(
    data=[
        go.Scatter(
            x=initial_gs_ss,
            y=initial_ba_ss,
            mode='lines',
            name='Bank Angle',
            line=dict(color='#e74c3c', width=3),
            hovertemplate='Groundspeed: %{x:.1f} kts<br>Bank Angle: %{y:.1f}°<extra></extra>'
        )
    ],
    layout=go.Layout(
        title=f'Bank Angle vs Groundspeed<br><sub>Max Differential: {max_diff:.2f}°</sub>',
        xaxis=dict(title='Groundspeed (knots)'),
        yaxis=dict(title='Bank Angle (degrees)'),
        template='plotly_white',
        height=450,
        margin=dict(l=60, r=20, t=80, b=60),
        hovermode='closest'
    )
)

# Create sliders
airspeed_slider_ss = widgets.FloatSlider(
    value=73, min=50, max=300, step=1,
    description='Airspeed (kts):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
windspeed_slider_ss = widgets.FloatSlider(
    value=10, min=1, max=100, step=1,
    description='Windspeed (kts):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
turn_radius_slider_ss = widgets.FloatSlider(
    value=1250, min=100, max=5000, step=10,
    description='Turn Radius (ft):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

def update_ss_plot(change):
    gs, ba = compute_steep_spiral_data(
        airspeed_slider_ss.value,
        windspeed_slider_ss.value,
        turn_radius_slider_ss.value
    )
    max_diff = max(ba) - min(ba)
    with ss_fig.batch_update():
        ss_fig.data[0].x = gs
        ss_fig.data[0].y = ba
        ss_fig.layout.title.text = f'Bank Angle vs Groundspeed<br><sub>Max Differential: {max_diff:.2f}°</sub>'

airspeed_slider_ss.observe(update_ss_plot, names='value')
windspeed_slider_ss.observe(update_ss_plot, names='value')
turn_radius_slider_ss.observe(update_ss_plot, names='value')

# Display sliders and figure separately (VS Code workaround)
display(widgets.VBox([
    airspeed_slider_ss,
    windspeed_slider_ss,
    turn_radius_slider_ss
], layout=widgets.Layout(margin='0 0 20px 0')))
display(ss_fig)

## Eights on Pylons

In [ ]:
# Eights on Pylons Visualization with Plotly

relative_wind_deg_eop = np.linspace(0, 180, 100)

def compute_pivotal_altitudes(airspeed, windspeed, msl_altitude):
    groundspeeds = [calculate_groundspeed(airspeed, windspeed, angle) for angle in relative_wind_deg_eop]
    altitudes = [calculate_pivotal_altitude(gs, msl_altitude) for gs in groundspeeds]
    return groundspeeds, altitudes

# Create initial figure
initial_gs_eop, initial_pa = compute_pivotal_altitudes(90, 10, 200)
max_diff_pa = max(initial_pa) - min(initial_pa)

eop_fig = go.FigureWidget(
    data=[
        go.Scatter(
            x=relative_wind_deg_eop,
            y=initial_pa,
            mode='lines',
            name='Pivotal Altitude',
            line=dict(color='#8e44ad', width=3),
            fill='tozeroy',
            fillcolor='rgba(142, 68, 173, 0.2)',
            customdata=initial_gs_eop,
            hovertemplate='Wind Angle: %{x:.0f}°<br>Groundspeed: %{customdata:.1f} kts<br>Pivotal Alt: %{y:.0f} ft<extra></extra>'
        )
    ],
    layout=go.Layout(
        title=f'Pivotal Altitude vs Relative Wind<br><sub>Max Differential: {max_diff_pa:.0f} ft</sub>',
        xaxis=dict(title='Relative Wind (degrees)', range=[0, 180]),
        yaxis=dict(title='Pivotal Altitude (feet)'),
        template='plotly_white',
        height=450,
        margin=dict(l=60, r=20, t=80, b=60),
        hovermode='x unified'
    )
)

# Create sliders
airspeed_slider_eop = widgets.FloatSlider(
    value=90, min=50, max=300, step=1,
    description='Airspeed (kts):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
windspeed_slider_eop = widgets.FloatSlider(
    value=10, min=0, max=100, step=1,
    description='Windspeed (kts):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
msl_altitude_slider_eop = widgets.FloatSlider(
    value=200, min=0, max=10000, step=50,
    description='MSL Alt (ft):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

def update_eop_plot(change):
    gs, pa = compute_pivotal_altitudes(
        airspeed_slider_eop.value,
        windspeed_slider_eop.value,
        msl_altitude_slider_eop.value
    )
    max_diff = max(pa) - min(pa)
    with eop_fig.batch_update():
        eop_fig.data[0].y = pa
        eop_fig.data[0].customdata = gs
        eop_fig.layout.title.text = f'Pivotal Altitude vs Relative Wind<br><sub>Max Differential: {max_diff:.0f} ft</sub>'

airspeed_slider_eop.observe(update_eop_plot, names='value')
windspeed_slider_eop.observe(update_eop_plot, names='value')
msl_altitude_slider_eop.observe(update_eop_plot, names='value')

# Display sliders and figure separately (VS Code workaround)
display(widgets.VBox([
    airspeed_slider_eop,
    windspeed_slider_eop,
    msl_altitude_slider_eop
], layout=widgets.Layout(margin='0 0 20px 0')))
display(eop_fig)

## Maneuvering Speed (Va)

The maneuvering speed $V_a$ varies with aircraft weight:

$$V_a = V_{a_{ref}} \times \sqrt{\frac{W}{W_{ref}}}$$

Where:
- $V_{a_{ref}} = 111$ KIAS at $W_{ref} = 2440$ lbs (max gross weight)
- At minimum weight (1531 lbs), $V_a = 88$ KIAS

In [ ]:
# Maneuvering Speed (Va) Calculator

EMPTY_WEIGHT = 1550  # lbs
VA_REF = 111         # KIAS at max gross
W_REF = 2440         # lbs (max gross)

def calculate_va(weight_lbs):
    """Calculate Va for a given weight using the square root of weight ratio."""
    return VA_REF * np.sqrt(weight_lbs / W_REF)

# Fuel selector dropdown
fuel_dropdown = widgets.Dropdown(
    options=[('Tab Fuel (35 gal / 210 lbs)', 210), ('Full Fuel (48 gal / 288 lbs)', 288)],
    value=288,
    description='Fuel:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Payload slider (pilot + passenger + baggage)
payload_slider = widgets.FloatSlider(
    value=340, min=0, max=600, step=1,
    description='Payload (lbs):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Weight breakdown display
weight_label = widgets.HTML(value='')

def get_total_weight():
    return EMPTY_WEIGHT + fuel_dropdown.value + payload_slider.value

# Create gauge
initial_va = calculate_va(get_total_weight())
va_gauge = go.FigureWidget(
    go.Indicator(
        mode="gauge+number",
        value=initial_va,
        number={'suffix': ' KIAS', 'font': {'size': 40}},
        gauge={
            'axis': {'range': [80, 120], 'tickwidth': 1, 'tickcolor': '#333'},
            'bar': {'color': '#8e44ad'},
            'bgcolor': 'white',
            'borderwidth': 2,
            'bordercolor': '#ccc',
            'steps': [
                {'range': [80, 90], 'color': '#e8f8f0'},
                {'range': [90, 100], 'color': '#d5e8d4'},
                {'range': [100, 110], 'color': '#ffeaa7'},
                {'range': [110, 120], 'color': '#fab1a0'}
            ],
        },
        title={'text': 'Maneuvering Speed (Va)', 'font': {'size': 20}}
    )
)
va_gauge.update_layout(
    template='plotly_white',
    height=350,
    margin=dict(l=20, r=20, t=80, b=20)
)

def update_va(change):
    total = get_total_weight()
    va = calculate_va(total)
    weight_label.value = (
        f'<b>Weight Breakdown:</b> Empty {EMPTY_WEIGHT} + '
        f'Fuel {fuel_dropdown.value} + Payload {payload_slider.value:.0f} = '
        f'<b>{total:.0f} lbs</b>'
    )
    with va_gauge.batch_update():
        va_gauge.data[0].value = va

fuel_dropdown.observe(update_va, names='value')
payload_slider.observe(update_va, names='value')

# Initialize label
update_va(None)

display(widgets.VBox([
    fuel_dropdown,
    payload_slider,
    weight_label
], layout=widgets.Layout(margin='0 0 20px 0')))
display(va_gauge)